<a href="https://colab.research.google.com/github/KeertanPatel-CS/movie_recommender-svd/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation System

## Milestone 2: Honest Testing and Baselines to Beat

**Team Members:** Kirtan Patel and Rudra Patel

**Course:** MATH 344L

In [34]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd

## Part A: Load the MovieLens Ratings

In [35]:
# Download the MovieLens 100K dataset
if not os.path.exists("ml-100k/u.data"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip"
    )

    with zipfile.ZipFile("ml-100k.zip", "r") as zip_ref:
        zip_ref.extractall()

In [36]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user", "item", "rating", "time"]
)

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


This code loads the MovieLens 100K dataset into a table called `ratings`. Each row represents one movie rating and includes the user ID, movie ID, rating, and time. The dataset contains 100,000 ratings in total.

## Part B: Make an Honest Train/Test Split

In [37]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    """Hide ratings without removing a user's or movie's last training rating."""
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)

print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


This code splits the movie ratings into training and testing data. About 90% of the ratings are used for training, and about 10% are hidden for testing.

It also makes sure that every user and movie still has at least one rating in the training data. This way, we can use the training ratings to make predictions and then compare them with the hidden ratings to see how accurate our recommendation system is.

## Part C: Build the Training Matrix

In [38]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


This code takes the training ratings and organizes them into a matrix called `R`. Each row represents a user, and each column represents a movie. If a user rated a movie, the matrix shows that rating. If they haven't rated it or the rating was hidden for testing, it shows `NaN`.

The matrix has 943 users and 1,682 movies. The `known` variable helps us identify which ratings are available in the training data.

In [39]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This code stores the actual hidden ratings in `truth`. The variables `u` and `i` store the user and movie positions for those ratings. We subtract 1 because the IDs start at 1, but Python indexing starts at 0.

Later, we'll use these values to compare our predictions with the actual ratings and calculate how accurate our models are.

## Part D: Score Predictions with RMSE

In [40]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

This code creates a function called `rmse` that measures how accurate our predicted movie ratings are compared to the actual ratings.

It finds the difference between each prediction and the actual rating, squares the differences, calculates the average, and then takes the square root.

Basically, a lower RMSE means our predictions are closer to the actual ratings, so the recommendation system is more accurate.

## Part E: Build Simple Baselines

In [41]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0

item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating]
)

print("Overall average:", round(mu, 3))

Overall average: 3.53


This code calculates the average movie ratings using only the training data. The variable `mu` stores the overall average rating, `user_mean` stores the average rating for each user, and `item_mean` stores the average rating for each movie.

The code ignores missing ratings (`NaN`) when calculating the averages. If a movie has no training ratings, it uses the overall average instead.

In [42]:
scores = {}

scores["always 3"] = rmse(
    np.full(len(test), 3.0), truth
)

scores["overall average"] = rmse(
    np.full(len(test), mu), truth
)

scores["user average"] = rmse(
    user_mean[u], truth
)

scores["movie average"] = rmse(
    item_mean[i], truth
)

for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


This code compares four simple ways of predicting movie ratings. The first model always predicts 3 stars, the second uses the overall average rating, the third uses each user's average rating, and the fourth uses each movie's average rating.

We calculate the RMSE for each model by comparing its predictions with the hidden test ratings. The model with the lowest RMSE makes the most accurate predictions on this test set.

## Part F: Combine User and Movie Effects

In [43]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)

    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]

    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0

    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating]
    )

    return mu, b_user, b_item

This code creates a function called `bias_baseline` that uses the overall average rating along with user and movie effects.

The `b_user` variable shows whether a user usually gives higher or lower ratings compared to the overall average. The `b_item` variable shows whether a movie gets higher or lower ratings after accounting for the user's rating habits.

Basically, this model combines both factors to make better predictions.

In [44]:
mu, b_user, b_item = bias_baseline(R)

pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores["user + movie effects"] = rmse(pred, truth)

print(pd.Series(scores, name="test RMSE").round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


This code uses the bias baseline model to predict the hidden movie ratings. It combines the overall average with the user and movie effects to calculate each prediction.

The `np.clip()` function makes sure that every predicted rating stays between 1 and 5 stars.

We then calculate the RMSE and compare it with the four previous models. The model with the lowest RMSE performs the best on the hidden test ratings.

## Practice Exercises

### Exercise 1: Calculate Bias Effects by Hand

The small matrix has six known ratings: 5, 3, 4, 1, 2, and 2.

**Overall Average:**

\[
\mu = \frac{5+3+4+1+2+2}{6} = \frac{17}{6} \approx 2.8333
\]

**User Effects:**

\[
b_{user,1} = 4 - 2.8333 = 1.1667
\]

\[
b_{user,2} = 2.5 - 2.8333 = -0.3333
\]

\[
b_{user,3} = 2 - 2.8333 = -0.8333
\]

**Movie Effects:**

After subtracting the overall average and user effects from the known ratings:

\[
b_{item,1} = (1+1.5)/2 = 1.25
\]

\[
b_{item,2} = (-1+0)/2 = -0.50
\]

\[
b_{item,3} = (-1.5+0)/2 = -0.75
\]

These values show how each user and movie differs from the overall average rating.

In [45]:
# Small ratings matrix from Exercise 1
tiny = np.array([
    [5, 3, np.nan],
    [4, np.nan, 1],
    [np.nan, 2, 2]
], dtype=float)

# Check our hand calculations
tiny_mu, tiny_b_user, tiny_b_item = bias_baseline(tiny)

print("Overall average:", round(tiny_mu, 4))
print("User effects:", np.round(tiny_b_user, 4))
print("Movie effects:", np.round(tiny_b_item, 4))

Overall average: 2.8333
User effects: [ 1.1667 -0.3333 -0.8333]
Movie effects: [ 1.25 -0.5  -0.75]


I calculated the overall average, user effects, and movie effects by hand first. Then I entered the small ratings matrix into Python using `np.nan` for the missing values and checked the results with the `bias_baseline()` function.

The Python results matched my hand calculations, which shows that the function is working correctly for this example.

### Exercise 2: See What Cheating Looks Like

In [46]:
# Build a matrix using all ratings, including hidden test ratings
R_all = to_matrix(ratings)

# Calculate user averages using all ratings
user_mean_all = np.nanmean(R_all, axis=1)

# Calculate the RMSE using the same test set
cheating_rmse = rmse(user_mean_all[u], truth)

# Compare honest and cheating RMSE scores
print("Honest user-average RMSE:", round(scores["user average"], 4))
print("Cheating user-average RMSE:", round(cheating_rmse, 4))

if cheating_rmse < scores["user average"]:
    print("The cheating RMSE is lower.")
else:
    print("The cheating RMSE is not lower.")

Honest user-average RMSE: 1.0426
Cheating user-average RMSE: 1.0322
The cheating RMSE is lower.


#### Exercise 2: Analysis

This code calculates the user averages using all the movie ratings, including the ones that were supposed to be hidden for testing.

The cheating RMSE may be lower than the honest RMSE because the model already used the test ratings when calculating the averages.

Basically, this is misleading because the model is supposed to make predictions without knowing the hidden ratings. Using those ratings gives the model an unfair advantage and doesn't show how well it would predict new ratings.

### Exercise 3: Is It Luck?

In [47]:
# Create a new split using seed=1
train_1, test_1 = split_ratings(ratings, seed=1)

# Build a new training matrix
R_1 = to_matrix(train_1)
known_1 = ~np.isnan(R_1)

# Prepare the new hidden test ratings
truth_1 = test_1["rating"].to_numpy()
u_1 = test_1["user"].to_numpy() - 1
i_1 = test_1["item"].to_numpy() - 1

print("Training ratings:", len(train_1))
print("Testing ratings:", len(test_1))

Training ratings: 90174
Testing ratings: 9826


This code creates another training and testing split using `seed=1` instead of `seed=0`. It also builds a new training matrix and stores the new hidden ratings so we can test the models again.

Basically, we're using a different random split to see if our recommendation models still perform the same way.

In [48]:
# Calculate averages using only the new training data
mu_1 = np.nanmean(R_1)
user_mean_1 = np.nanmean(R_1, axis=1)

n_item_1 = known_1.sum(axis=0)
item_mean_1 = np.full(N_ITEMS, mu_1)
has_rating_1 = n_item_1 > 0

item_mean_1[has_rating_1] = (
    np.nansum(R_1, axis=0)[has_rating_1] / n_item_1[has_rating_1]
)

# Calculate the five RMSE scores
scores_1 = {}

scores_1["always 3"] = rmse(
    np.full(len(test_1), 3.0), truth_1
)

scores_1["overall average"] = rmse(
    np.full(len(test_1), mu_1), truth_1
)

scores_1["user average"] = rmse(
    user_mean_1[u_1], truth_1
)

scores_1["movie average"] = rmse(
    item_mean_1[i_1], truth_1
)

# Calculate the user + movie effects model
mu_bias_1, b_user_1, b_item_1 = bias_baseline(R_1)

pred_1 = np.clip(
    mu_bias_1 + b_user_1[u_1] + b_item_1[i_1],
    1, 5
)

scores_1["user + movie effects"] = rmse(pred_1, truth_1)

# Compare both splits
comparison = pd.DataFrame({
    "Seed 0 RMSE": pd.Series(scores),
    "Seed 1 RMSE": pd.Series(scores_1)
})

comparison["Difference"] = (
    comparison["Seed 1 RMSE"] - comparison["Seed 0 RMSE"]
)

print(comparison.round(4))

                      Seed 0 RMSE  Seed 1 RMSE  Difference
always 3                   1.2516       1.2336     -0.0180
overall average            1.1349       1.1127     -0.0222
user average               1.0426       1.0345     -0.0081
movie average              1.0284       1.0159     -0.0125
user + movie effects       0.9545       0.9577      0.0032


In [49]:
# Compare model rankings
ranking_0 = sorted(scores, key=scores.get)
ranking_1 = sorted(scores_1, key=scores_1.get)

print("Seed 0 ranking:", ranking_0)
print("Seed 1 ranking:", ranking_1)

if ranking_0 == ranking_1:
    print("The model rankings stayed the same.")
else:
    print("The model rankings changed.")

Seed 0 ranking: ['user + movie effects', 'movie average', 'user average', 'overall average', 'always 3']
Seed 1 ranking: ['user + movie effects', 'movie average', 'user average', 'overall average', 'always 3']
The model rankings stayed the same.


#### Exercise 3: Analysis

When I changed the seed from 0 to 1, the training and testing ratings changed, so the RMSE scores can also change.

This happens because the models are being tested on a different set of hidden ratings.

Basically, this shows that we shouldn't decide which recommendation model is better based on just one random split. We should test the models using multiple splits to make sure the results are consistent.